# Lekcja 16: Statystyka i prawdopodobieństwo — Rozwiązania zadań

Pełne treści zadań z podręcznika + rozwiązania.
Uruchom po przejściu lekcji.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import (
    shapiro, pearsonr, spearmanr, chi2_contingency,
    binom, ttest_1samp, ttest_ind, levene, f_oneway
)
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.outliers_influence import variance_inflation_factor
import time
%matplotlib inline
np.random.seed(42)

# Datasety
IRIS_URL    = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv'
TITANIC_URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
iris    = pd.read_csv(IRIS_URL)
titanic = pd.read_csv(TITANIC_URL)

# Syntetyczny Housing (fetch_california_housing wymaga połączenia)
np.random.seed(42); n_h = 2000
MedInc  = np.random.exponential(3.5, n_h).clip(0.5, 15)
AveRms  = np.random.normal(5.4, 2.2, n_h).clip(1, 25)
HouseAge= np.random.uniform(1, 52, n_h)
Lat     = np.random.uniform(32.5, 42, n_h)
Lon     = np.random.uniform(-124, -114, n_h)
Popul   = np.random.exponential(1500, n_h).clip(50, 35000)
AveOcc  = np.random.normal(3.0, 1.5, n_h).clip(1, 15)
MedHVal = (0.6*MedInc + 0.05*AveRms - 0.005*HouseAge
           + np.random.normal(0, 0.4, n_h)).clip(0.5, 5)
housing = pd.DataFrame({'MedInc':MedInc,'AveRooms':AveRms,'HouseAge':HouseAge,
                        'Latitude':Lat,'Longitude':Lon,
                        'Population':Popul,'AveOccup':AveOcc,
                        'MedHouseVal':MedHVal})

# Syntetyczne Wine Quality
np.random.seed(42); n_w = 1599
alc   = np.random.normal(10.42, 1.07, n_w).clip(8.4, 14.9)
vacid = np.random.normal(0.528, 0.179, n_w).clip(0.12, 1.58)
sulph = np.random.normal(0.658, 0.170, n_w).clip(0.33, 2.0)
qual_raw = 3 + 0.4*alc - 1.5*vacid + 0.5*sulph + np.random.normal(0, 0.8, n_w)
wine = pd.DataFrame({
    'fixed acidity':         np.random.normal(8.32, 1.74, n_w).clip(4.6, 15.9),
    'volatile acidity':      vacid,
    'citric acid':           np.random.normal(0.271, 0.195, n_w).clip(0, 1),
    'residual sugar':        np.random.exponential(2.5, n_w).clip(1.2, 15.5),
    'chlorides':             np.random.normal(0.087, 0.047, n_w).clip(0.012, 0.611),
    'free sulfur dioxide':   np.random.exponential(15, n_w).clip(1, 72),
    'total sulfur dioxide':  np.random.normal(46.5, 32.9, n_w).clip(6, 289),
    'density':               np.random.normal(0.9967, 0.002, n_w).clip(0.990, 1.004),
    'pH':                    np.random.normal(3.311, 0.154, n_w).clip(2.74, 4.01),
    'sulphates':             sulph,
    'alcohol':               alc,
    'quality':               np.round(np.clip(qual_raw, 3, 8)).astype(int)
})

print(f'Iris: {iris.shape} | Titanic: {titanic.shape}')
print(f'Housing: {housing.shape} | Wine: {wine.shape}')

Iris: (150, 5) | Titanic: (891, 12)
Housing: (2000, 8) | Wine: (1599, 12)


---
## ✏️ Zadania podstawowe (1–8)

### ✏️ Zadanie 1 – Generowanie rozkładu normalnego

Wygeneruj 1000 próbek z rozkładu normalnego N(μ=50, σ=10). Narysuj histogram
i sprawdź, czy ~68% wartości mieści się w przedziale [μ-σ, μ+σ].

**Wymagania:**
- Oblicz empiryczny procent wartości w przedziale [40, 60]
- Porównaj z teoretycznym 68.27%
- Narysuj histogram z zaznaczonym przedziałem ±1σ

*(proste)*

### ✏️ Zadanie 2 – Obliczanie prawdopodobieństw

Dana jest zmienna losowa X~N(100, 15). Oblicz prawdopodobieństwa:
P(X < 85), P(X > 120), P(90 < X < 110). Wizualizuj obszary na krzywej rozkładu.

*(proste)*

In [5]:
prawdopodobienstwo_85 = stats.norm.cdf(85, loc=100, scale=15)
print(f'P(X < 85) = {prawdopodobienstwo_85:.4f}')

prawdopodobienstwo_120 = 1 - stats.norm.cdf(120, loc=100, scale=15)
print(f'P(X > 120) = {prawdopodobienstwo_120:.4f}')

prawdopodobienstwo_90_110 = stats.norm.cdf(110, loc=100, scale=15) - stats.norm.cdf(90, loc=100, scale=15)
print(f'P(90 < X < 110) = {prawdopodobienstwo_90_110:.4f}')

P(X < 85) = 0.1587
P(X > 120) = 0.0912
P(90 < X < 110) = 0.4950


### ✏️ Zadanie 3 – Analiza rozkładu Iris

Sprawdź normalność rozkładu cechy `sepal_length` testem Shapiro-Wilka.
Narysuj histogram + KDE i Q-Q plot. Zinterpretuj wynik.

*(proste)*

### ✏️ Zadanie 4 – Korelacja Pearsona

Oblicz korelację Pearsona między `petal_length` a `petal_width` w datasecie Iris.
Narysuj scatter plot z linią regresji.

*(proste)*

### ✏️ Zadanie 5 – Macierz korelacji

Narysuj heatmap macierzy korelacji dla datasetu California Housing.
Zidentyfikuj 3 pary cech o najwyższej korelacji (poza targetem).

*(proste)*

In [1]:
corr = housing.corr(numeric_only=True)

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title("Macierz korelacji")
plt.show()

NameError: name 'housing' is not defined

In [23]:
corr_no_target = corr.drop(index='MedHouseVal', columns='MedHouseVal')

pairs = (
    corr_no_target.where(~np.tril(np.ones(corr_no_target.shape), k=0).astype(bool))
    .stack()
    .sort_values(key=abs, ascending=False)
)

print(pairs.head(3))

Latitude  AveOccup    0.036286
MedInc    HouseAge    0.033142
HouseAge  AveOccup   -0.027709
dtype: float64


### ✏️ Zadanie 6 – Test t jednej próbki

Dla datasetu Titanic przetestuj hipotezę, że średni wiek pasażerów wynosi 30 lat.
H₀: μ = 30

*(proste)*

In [7]:
titanic_test = titanic.copy()

t_stat, p_value = stats.ttest_1samp(
    titanic_test['Age'].dropna(),
    popmean=30
)

print(f"""
t = {t_stat:.4f}
p_value = {p_value:.4f}""")


t = -0.5535
p_value = 0.5801


### ✏️ Zadanie 7 – Test t dwóch prób

Porównaj średni wiek pasażerów Titanica którzy przeżyli vs nie przeżyli.
Czy różnica jest istotna statystycznie?

*(proste)*

In [11]:
titanic_2sample = titanic.copy()

survived = titanic_2sample[titanic_2sample['Survived'] == 1]['Age'].dropna()
not_survived = titanic_2sample[titanic_2sample['Survived'] == 0]['Age'].dropna()

# Test Levene'a
levene_stat, levene_p = stats.levene(survived, not_survived)

print(f"Levene statistic = {levene_stat:.4f}")
print(f"Levene p-value = {levene_p:.4f}")

equal_var = levene_p > 0.05

t_stat, p_value = stats.ttest_ind(
    survived,
    not_survived,
    equal_var=equal_var
)

print(f"\nt = {t_stat:.4f}")
print(f"p-value = {p_value:.4f}")

if p_value < 0.05:
    print("\nOdrzucamy H₀ - średni wiek pasażerów różni się istotnie statystycznie.")
else:
    print("\nBrak podstaw do odrzucenia H₀ - średni wiek pasażerów nie różni się istotnie statystycznie.")

Levene statistic = 1.1954
Levene p-value = 0.2746

t = -2.0667
p-value = 0.0391

Odrzucamy H₀ - średni wiek pasażerów różni się istotnie statystycznie.


### ✏️ Zadanie 8 – Test chi-kwadrat

Przetestuj, czy przeżycie na Titanicu zależy od klasy podróży (Pclass).
Narysuj stacked bar chart z procentami. Oblicz Cramer's V.

*(proste)*

In [14]:
titanic_survival_pclass = titanic.copy()

# H0 przeżycie zależy od klasy, H1 przeżycie nie zależy od klasy
contingency = pd.crosstab(
    titanic_survival_pclass['Pclass'].dropna(),
    titanic_survival_pclass['Survived'].dropna(),
)

chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
print(f"Chi2 = {chi2:.4f}")
print(f"p-value = {p_value:.4f}")
print(f"Degrees of freedom = {dof}")

n = contingency.to_numpy().sum()
r, c = contingency.shape

cramers_v = np.sqrt(chi2 / (n * (min(r - 1, c - 1))))

print(f"Cramer's V = {cramers_v:.4f}")

Chi2 = 102.8890
p-value = 0.0000
Degrees of freedom = 2
Cramer's V = 0.3398


---
## ✏️ Zadania średnie (9–12)

### ✏️ Zadanie 9 – Rozkład dwumianowy — symulacja

Symuluj 1000 eksperymentów rzutu 20 monetami (p=0.5). Porównaj histogram
wyników z teoretycznym rozkładem dwumianowym B(20, 0.5).

*(średnie)*

### ✏️ Zadanie 10 – Wykrywanie outlierów metodą z-score

Dla kolumny `alcohol` w datasecie Wine Quality zidentyfikuj outliery metodą
z-score (|z| > 3). Usuń je i porównaj statystyki przed/po.

*(średnie)*

### ✏️ Zadanie 11 – Korelacja Pearson vs Spearman

Stwórz 3 syntetyczne datasety (zależność liniowa, kwadratowa, z outlierami).
Oblicz i porównaj Pearsona i Spearmana dla każdego.

*(średnie)*

### ✏️ Zadanie 12 – ANOVA — porównanie wielu grup

Porównaj średnią długość płatka (`petal_length`) między trzema gatunkami Iris.
Jeśli wynik istotny, wykonaj testy post-hoc (Tukey HSD).

*(średnie)*

---
## 🧠 Zadania wyzwanie (13–20)

### 🧠 Zadanie 13 – Symulacja Central Limit Theorem

Wygeneruj 10 000 próbek rozmiaru n=30 z rozkładu jednostajnego U(0, 10).
Dla każdej próbki oblicz średnią. Sprawdź, czy rozkład średnich jest normalny.

*(challenge)*

### 🧠 Zadanie 14 – Bootstrapping — przedziały ufności

Oblicz 95% przedział ufności dla średniego wieku pasażerów Titanica metodą
bootstrap (10 000 próbek z replacement).

*(challenge)*

### 🧠 Zadanie 15 – Power analysis dla A/B testu

Zaplanuj A/B test: wykryj wzrost konwersji z 10% do 12% z mocą 80% i α=0.05.
Oblicz wymaganą wielkość próby per grupę. Potwierdź symulacją.

*(challenge)*

### 🧠 Zadanie 16 – Multikolinearność i VIF

Dla datasetu Wine Quality oblicz VIF dla każdej cechy. Usuń iteracyjnie cechy
z VIF > 10. Porównaj macierze korelacji przed i po.

*(challenge)*

### 🧠 Zadanie 17 – Permutation test

Porównaj średnią długość płatka setosa vs versicolor używając permutation test
(10 000 permutacji). Porównaj p-value z klasycznym t-testem.

*(challenge)*

### 🧠 Zadanie 18 – Multiple testing correction (Bonferroni)

Przetestuj korelację między `quality` a każdą z 11 cech Wine Quality.
Zastosuj korekcję Bonferroniego. Które korelacje pozostają istotne?

*(challenge)*

### 🧠 Zadanie 19 – Analiza rozkładów z dopasowaniem

Dopasuj 3 rozkłady do kolumny Fare z Titanica (normalny, lognormalny, gamma).
Wybierz najlepszy używając testu Kołmogorowa-Smirnowa.

*(challenge)*

### 🧠 Zadanie 20 – Bayesian A/B test

Zasymuluj A/B test (n=500 per grupę, p_A=0.10, p_B=0.12).
Oblicz P(p_B > p_A) metodą Beta-Binomial. Porównaj z klasycznym testem.

*(challenge)*